# 📦 ITEC Product Re-categorization — Full Pipeline

> จัดหมวดสินค้า ITEC ใหม่ทั้งหมด แก้ปัญหา category เดิม **key ผิด / ซ้ำซ้อน**
> Feature: `ItemName` · Target: หมวดที่คุณตั้งเอง · Backbone: LLM embedding (ไม่ fine-tune)

---

## 🗺️ ภาพรวมขั้นตอนสร้างโมเดล (อ่านก่อนรัน)

```
ItemName ดิบ
   │
① Normalize          ทำความสะอาดข้อความ                         [Cell 3]
   │
② สร้าง Flags         คำใน ItemName → IS_ flags (keyword)         [Cell 4-5] ✏️
   │                  เช่น "iphone" → IS_iPhone=1
   │
③ จัดหมวด (Label)     รวม flags → หมวดใหญ่ = y                    [Cell 6] ✏️
   │                  เช่น IS_Case → "อุปกรณ์เสริม"
   │
④ Embedding          LLM อ่าน ItemName → เวกเตอร์ (แช่แข็ง)       [Cell 8]
   │
⑤ เทรน Classifier     LogisticRegression: เวกเตอร์ → หมวด         [Cell 9-10]
   │                  ★ ส่วนที่ "เรียน" จริง มีแค่ตรงนี้
   │
⑥ ประเมิน            Accuracy + Macro-F1                        [Cell 11]
   │
⑦ ทำ label สะอาด      จับ item ที่ key ผิด → แก้ → retrain         [Cell 12]
   │
⑧ (option) Fine-tune  ปรับ weights LLM — ทำท้ายสุด ถ้ายังไม่พอ    [Cell 13]
```

## 🎚️ 3 ระดับที่คุณตั้งเองได้

| ระดับ | คือ | แก้ที่ |
|---|---|---|
| 1. **Flag** (ชื่ออังกฤษ) | มี flag อะไรบ้าง | Cell 4 ✏️ |
| 2. **คำใน flag** (ไทย/eng) | keyword ที่ทำให้ติด flag | Cell 4 ✏️ |
| 3. **จัดหมวด flag** (label) | flag ไหนรวมเป็นหมวดใหญ่ | Cell 6 ✏️ |

## ⚠️ ลำดับความคุ้ม (ทำตามนี้)

```
1. embedding + classifier (baseline)  ← ทำก่อนเสมอ
2. ทำ label สะอาด + เพิ่มข้อมูล        ← คุ้มสุด
3. fine-tune                          ← ท้ายสุด · ดีขึ้น "ต่อเมื่อ label สะอาดก่อน"
```

> 🔑 fine-tune บน label ที่ยัง noisy = เรียนของผิดแม่นขึ้น (แย่ลง) → ต้องทำ label สะอาดก่อน

## 📌 หมายเหตุ
- นี่**ไม่ใช่ fine-tune** — LLM แช่แข็ง เรียนแค่ LogisticRegression
- 2 โมเดลเทียบ: M1 random-drop (predict ItemName ล้วน) · M2 full (predict ครบ)
- CPU ช้า → ใช้ subset / MiniLM · GPU → bge-m3 (ดูตาราง performance ใน Obsidian)
- ✏️ = cell ที่ต้องแก้ · cell อื่นรันผ่านได้เลย

## 1 · Setup — รันได้ทั้ง local · Colab · Kaggle

ตรวจสภาพแวดล้อมเองแล้วหาไฟล์ให้ ไม่ต้องแก้อะไร

| ที่ไหน | ต้องเตรียมอะไร |
|---|---|
| **เครื่องตัวเอง** | ไม่ต้อง — โน้ตบุ๊กอยู่โฟลเดอร์เดียวกับ `_data/` แล้ว |
| **Colab** | Cell นี้จะขอ `dim_item_itec.csv.gz` แล้วขอ `itec_category_scripts.zip` |
| **Kaggle** | **Add Data → Upload** ทั้งสองไฟล์เป็น Dataset เดียว แล้ว Cell นี้คัดลอกออกมาให้เอง |

> **Kaggle:** Settings → Accelerator → **GPU T4 x2** · ฟรี 30 ชม./สัปดาห์ (คนละโควตากับ Colab)


In [2]:
import sys, os, subprocess, glob, gzip, shutil, zipfile
from pathlib import Path

# ---------------------------------------------------------------- ที่ไหน --
IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/input").exists()
ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"

try:
    import sentence_transformers
except ImportError:
    print("ติดตั้ง sentence-transformers ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "sentence-transformers"], check=True)


def _ungz(src, dst):
    with gzip.open(src, "rb") as f, open(dst, "wb") as o:
        shutil.copyfileobj(f, o)


if ENV == "local":
    # โน้ตบุ๊กอยู่โฟลเดอร์เดียวกับสคริปต์และ _data อยู่แล้ว
    SCRIPTS = Path.cwd()

elif ENV == "colab":
    SCRIPTS = Path("/content")
    os.chdir(SCRIPTS)
    if not Path("dim_item_itec.csv").exists():
        print("เลือกไฟล์  dim_item_itec.csv.gz  (5.8 MB)")
        from google.colab import files
        files.upload()
        if Path("dim_item_itec.csv.gz").exists():
            _ungz("dim_item_itec.csv.gz", "dim_item_itec.csv")
    if not Path("type_platform.py").exists():
        print("เลือกไฟล์  itec_category_scripts.zip  (22 KB)")
        from google.colab import files
        files.upload()
        zipfile.ZipFile("itec_category_scripts.zip").extractall(".")

else:  # kaggle - ไฟล์อยู่ใน /kaggle/input แบบอ่านอย่างเดียว ต้องคัดลอกออกมาก่อน
    SCRIPTS = Path("/kaggle/working")
    os.chdir(SCRIPTS)
    # ไล่ทุกชั้น เอาเฉพาะไฟล์ - dataset อาจซ้อนโฟลเดอร์หลายชั้น
    src = [p for p in Path("/kaggle/input").rglob("*") if p.is_file()]
    print("เจอใน /kaggle/input:", [p.name for p in src][:20])
    for p in src:
        if p.suffix == ".zip":
            zipfile.ZipFile(p).extractall(".")
        elif p.name.endswith(".csv.gz"):
            _ungz(p, p.name[:-3])
        elif not Path(p.name).exists():
            shutil.copy(p, p.name)

# หาไฟล์ข้อมูล - local อยู่ใน _data/ ที่เหลืออยู่ข้าง ๆ
DATA = next((p for p in (SCRIPTS / "_data" / "dim_item_itec.csv",
                         SCRIPTS / "dim_item_itec.csv") if p.exists()), None)
assert DATA, f"ไม่พบ dim_item_itec.csv ใน {SCRIPTS} · ที่มี: {[p.name for p in SCRIPTS.iterdir()][:12]}"

sys.path.insert(0, str(SCRIPTS))
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
import re, numpy as np, pandas as pd, joblib

try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
          else "ไม่มี - จะใช้ CPU + MiniLM")
except ImportError:
    print("ยังไม่มี torch")
print(f"env={ENV} · SCRIPTS={SCRIPTS} · DATA={DATA.name} · pandas {pd.__version__}")


เจอใน /kaggle/input: ['dim_item_itec.csv', 'extract_keywords.py', 'type_platform.yaml', 'rules.py', 'benchmark_models.py', 'itec_mapper.py', 'predict_all.py', 'type_platform.py', 'item_keywords.yaml', 'explore_flags.py']
GPU: Tesla T4
env=kaggle · SCRIPTS=/kaggle/working · DATA=dim_item_itec.csv · pandas 2.3.3


## 2 · ✏️ โหลดข้อมูล — ต้องมี ItemName, CategoryName, SubCategoryName, Brand

In [3]:
df = pd.read_csv(DATA, encoding="utf-8-sig", dtype=str).fillna("")

# CSV ที่ export มาใช้ snake_case ต้อง map เป็นชื่อที่ notebook นี้ใช้
RENAME = {"item_id": "ItemId", "item_name": "ItemName", "category": "CategoryName",
          "sub_category": "SubCategoryName", "brand": "Brand"}
df = df.rename(columns={k: v for k, v in RENAME.items() if k in df.columns})

for col in ["CategoryName", "SubCategoryName", "Brand"]:
    if col not in df.columns: df[col] = ""

assert "ItemName" in df.columns, f"ไม่พบคอลัมน์ ItemName · ที่มีคือ {list(df.columns)}"
print("rows:", len(df)); df.head()


rows: 216009


,ItemId,base_item_id,ItemName,department,CategoryName,SubCategoryName,model_series,Brand,brand_group,product_status,is_valid_item_id
0,** CASE NIPDA 5161 VIVA WW 450 W. SATA,** CASE NIPDA 5161 VIVA WW 450 W. SATA,CASE NIPDA 5161 VIVA WW 45OW.,PC Components,PC Case & Cooling,CASE,,NIPDA,Other,ACTIVE,false
1,.GLOBAL A810,.GLOBAL A810,vv Headphone GLOBAL A810,Audio,Headphone,HEADPHONE,,GLOBAL,Other,ACTIVE,false
2,0 50644 51660 3,0 50644 51660 3,^^ Monster AI 800 MINI-3(Mini jack to Mini jac...,Accessories,Phone Accessories,CABLE,,MONSTER,Other,ACTIVE,false
3,0 50644 51663 4,0 50644 51663 4,^^ Monster AI 1000 Y - SPLT(Y-Splitter),Accessories,Phone Accessories,CABLE,,MONSTER,Other,ACTIVE,false
4,0%-10M-CASH-BACK-6780,0%-10M-CASH-BACK-6780,โปรโมชั่นผ่อน 0% 10เดือน Samsung ช่วยออก 2 เดื...,Other,รายการส่งเสริมการขาย,OTHER,,COM7,Other,ACTIVE,true


## 3 · Normalize

นิยามฟังก์ชันไว้ก่อน · **ยังไม่สร้าง `NORM`** เพราะต้องรอกรองหมวดที่ Cell 7 ก่อน

> ⚠️ **อย่าเรียก `normalize(pd.Series([x]))` ทีละแถว** — วัดแล้วช้ากว่า 563 เท่า


In [4]:
def normalize(s: pd.Series) -> pd.Series:
    """ทำความสะอาดข้อความ · ก-๛ ต้องมี ไม่งั้น pandas 3 + RE2 ลบภาษาไทยทิ้งหมด"""
    return (s.fillna("").astype(str).str.lower()
             .str.replace(r"[^\w\s/&+.\-ก-๛]", " ", regex=True)
             .str.replace(r"\s+", " ", regex=True).str.strip())

TEXT_COLS = ["ItemName", "CategoryName", "SubCategoryName", "Brand"]

df[["ItemName"]].assign(normalized=normalize(df.ItemName)).head()


,ItemName,normalized
0,CASE NIPDA 5161 VIVA WW 45OW.,case nipda 5161 viva ww 45ow.
1,vv Headphone GLOBAL A810,vv headphone global a810
2,^^ Monster AI 800 MINI-3(Mini jack to Mini jac...,monster ai 800 mini-3 mini jack to mini jack 3...
3,^^ Monster AI 1000 Y - SPLT(Y-Splitter),monster ai 1000 y - splt y-splitter
4,โปรโมชั่นผ่อน 0% 10เดือน Samsung ช่วยออก 2 เดื...,โปรโมชั่นผ่อน 0 10เดือน samsung ช่วยออก 2 เดือ...


## 4 · ✏️ กฎแยกประเภท + แบรนด์ — **แก้ที่นี่ที่เดียว**

รัน cell นี้แล้วกฎจะถูกเขียนลง `type_platform.yaml` ให้สคริปต์ใช้ต่ออัตโนมัติ

| อยากทำ | แก้ตรงไหน |
|---|---|
| เพิ่มคำ | เติมใน `"words"` |
| เพิ่มประเภทใหม่ | เพิ่มบรรทัด `{"label": ..., "words": [...], "whole_word": ...}` |
| **เปลี่ยนลำดับความสำคัญ** | **ย้ายบรรทัดขึ้นลง** — ตัวแรกที่ตรงชนะ |
| เปลี่ยนชื่อหมวดเป็นไทย | แก้ `"label"` |
| กันคำสั้นจับมั่ว | `"whole_word": True` |
| ลบประเภท | ลบทั้งบรรทัด |


In [ ]:
# ============================================================================
# ✏️ ที่เดียวที่ต้องแก้กฎ — แก้ตรงนี้แล้วรัน cell นี้ YAML จะถูกเขียนใหม่ให้เอง
# ============================================================================
# ลำดับ = ความสำคัญ · ตัวแรกที่ตรงชนะ · ย้ายขึ้นลงเพื่อเปลี่ยนลำดับ
# whole_word True = ต้องเป็นคำเต็ม เช่น stand จะไม่ไปจับ standard
#
# 🔑 กฎแบ่งเป็น 4 ชั้น ห้ามสลับชั้น (สลับคำในชั้นเดียวกันได้)
#    1 ไม่ใช่ตัวสินค้า   ต้องจับก่อน ไม่งั้นโปรโมชั่นจะกลายเป็นมือถือ
#    2 ชิ้นส่วนคอม
#    3 อุปกรณ์เสริม     ต้องชนะตัวเครื่อง — "เคส iPhone" คือเคส ไม่ใช่ iPhone
#    4 ตัวเครื่อง        ชั้นนี้ยังทำหน้าที่แปลง category เป็นชนิดไปในตัว
#                       เพราะ classify() เอา context มาไล่ TYPES ชุดเดียวกันซ้ำ
#                       เมื่อชื่อสินค้าไม่บอกอะไรเลย → คำอย่าง notebook/monitor/
#                       printer จึงต้องมีอยู่ ถึงจะดูด category มาใช้ได้

TYPES = [
    # ── ชั้น 1 · ไม่ใช่ตัวสินค้า ─────────────────────────────────────────
    {"label": "Warranty", "words": ["applecare", "care+", "ประกัน", "insurance", "warranty", "คุ้มครอง"], "whole_word": False},
    {"label": "Telecom", "words": ["promotion", "โปรโมชั่น", "เปิดเบอร์", "รายเดือน", "เติมเงิน", "trade up", "walkin", "prebook", "แพ็กเกจ", "ย้ายค่าย", "รายการส่งเสริมการขาย", "แลกซื้อ"], "whole_word": False},
    {"label": "Service", "words": ["ค่าบริการ", "ค่าส่ง", "ค่าแรง", "ค่าขนส่ง", "ค่าติดตั้ง", "shipping", "ซ่อม", "repair", "fee", "service charge", "ค่าธรรมเนียม"], "whole_word": True},
    # แยก part/spare ออกจาก Service — "(Part) LCD" คืออะไหล่ ไม่ใช่ค่าบริการ
    {"label": "SparePart", "words": ["อะไหล่", "spare", "part", "after sales", "แพรจอ", "ชุดจอ"], "whole_word": True},

    # ── ชั้น 2 · ชิ้นส่วนคอม ────────────────────────────────────────────
    {"label": "PowerSupply", "words": ["power supply", "psu ", "เพาเวอร์ซัพพลาย"], "whole_word": False},
    # เอา "aio " ออก — มันไปจับ "DESKTOP AIO" และ "Vaio" รวม 1,517 แถว
    {"label": "Cooling", "words": ["cooling", "cooler", "heatsink", "พัดลม", "ระบายความร้อน", "liquid freezer"], "whole_word": False},
    {"label": "Mainboard", "words": ["mainboard", "motherboard", "เมนบอร์ด", "m/b "], "whole_word": False},
    {"label": "GraphicCard", "words": ["graphic card", "vga ", "geforce", "radeon", "rtx ", "gtx "], "whole_word": False},
    {"label": "RAM", "words": ["ddr4", "ddr5", "so-dimm", "udimm"], "whole_word": False},
    {"label": "RAM", "words": ["ram"], "whole_word": True},
    {"label": "CPU", "words": ["ryzen", "core i3", "core i5", "core i7", "core i9", "ultra 5", "ultra 7", "ultra 9"], "whole_word": False},

    # ── ชั้น 3 · อุปกรณ์เสริม ต้องอยู่เหนือตัวเครื่องเสมอ ──────────────────
    {"label": "Bag", "words": ["bag", "กระเป๋า", "sleeve", "pouch", "backpack", "เป้"], "whole_word": True},
    {"label": "Case", "words": ["case", "casing", "เคส", "cover", "ฝาหลัง", "ฝาครอบ", "ซองใส่", "bumper", "กันกระแทก", "skin"], "whole_word": True},
    {"label": "Case", "words": ["protec"], "whole_word": False},
    {"label": "Film", "words": ["film", "ฟิล์ม", "tempered", "กระจกนิรภัย"], "whole_word": True},
    # Memory ต้องมาก่อน Charger — "MicroSDHC with SD Adapter" ไม่ใช่ที่ชาร์จ
    {"label": "Memory", "words": ["micro sd", "microsd", "sdhc", "sdxc", "memory card", "flash drive", "flashdrive", "thumb drive", "memory stick"], "whole_word": False},
    {"label": "Adapter", "words": ["card reader", "usb hub", "docking station", "แปลงสัญญาณ"], "whole_word": False},
    {"label": "Charger", "words": ["charger", "adapter", "อะแดปเตอร์", "ที่ชาร์จ", "หัวชาร์จ", "power bank", "powerbank", "พาวเวอร์แบงค์", "ปลั๊ก"], "whole_word": False},
    {"label": "Cable", "words": ["cable", "สายชาร์จ", "สาย usb", "lightning", "type-c", "type c"], "whole_word": False},
    {"label": "Strap", "words": ["strap", "สายนาฬิกา", "สายรัด", "band"], "whole_word": True},
    {"label": "Stand", "words": ["stand", "ขาตั้ง", "holder", "ที่วาง", "dock", "แท่นวาง"], "whole_word": True},
    {"label": "Keyboard", "words": ["keyboard", "คีย์บอร์ด"], "whole_word": False},
    {"label": "Mouse", "words": ["mouse", "เมาส์", "mousepad"], "whole_word": True},

    # ── ชั้น 4 · ตัวเครื่อง ── เพิ่มใหม่ทั้งชั้น · ตัวที่ดึง Device ลงจาก 58% ──
    {"label": "Smartwatch", "words": ["smartwatch", "watch series", "watch ultra", "watch se", "galaxy watch", "watch fit", "นาฬิกาอัจฉริยะ"], "whole_word": False},
    {"label": "Notebook", "words": ["notebook", "laptop", "macbook", "vaio", "thinkpad", "ideapad", "ultrabook", "chromebook", "surface laptop", "surface pro", "โน๊ตบุ๊ค", "โน้ตบุ๊ก"], "whole_word": False},
    {"label": "Desktop", "words": ["desktop", "all-in-one", "computer set", "pc set", "imac", "mac mini", "mac studio", "mac pro", "คอมประกอบ"], "whole_word": False},
    # ⚠️ "mac" เดี่ยว ๆ ต้องอยู่หลัง Desktop ไม่งั้น "Mac mini"/"Mac Studio"
    #    จะโดน mac จับไปเป็น Notebook ก่อน (เจอจริงตอนเทียบกับ prototype)
    {"label": "Notebook", "words": ["mac", "nb"], "whole_word": True},
    {"label": "Desktop", "words": ["aio", "pc"], "whole_word": True},
    # Tablet ต้องมาก่อน Network — "iPad Wi-Fi + Cellular" ไม่ใช่เราเตอร์
    {"label": "Tablet", "words": ["tablet", "ipad", "แท็บเล็ต", "galaxy tab"], "whole_word": False},
    {"label": "Smartphone", "words": ["smartphone", "สมาร์ทโฟน", "มือถือ", "โทรศัพท์"], "whole_word": False},
    {"label": "Monitor", "words": ["monitor", "จอมอนิเตอร์", "จอคอม"], "whole_word": False},
    {"label": "TV", "words": ["ทีวี", "โทรทัศน์", "bravia", "smart tv"], "whole_word": False},
    {"label": "TV", "words": ["tv"], "whole_word": True},
    {"label": "Printer", "words": ["printer", "เครื่องพิมพ์", "ปริ้นเตอร์", "toner", "หมึกพิมพ์", "ตลับหมึก", "inkjet", "laserjet"], "whole_word": False},
    {"label": "Camera", "words": ["camera", "กล้อง", "webcam", "dslr", "mirrorless", "handycam", "gopro"], "whole_word": False},
    {"label": "Storage", "words": ["ssd", "hdd", "hard drive", "harddisk", "hard disk", "usb drive", "external drive", "nas ", "optical drive", "handy drive", "storage"], "whole_word": False},
    {"label": "Audio", "words": ["headphone", "หูฟัง", "earphone", "earbud", "speaker", "ลำโพง", "soundbar", "headset", "airpod"], "whole_word": False},
    {"label": "Audio", "words": ["buds"], "whole_word": True},
    {"label": "Console", "words": ["playstation", "nintendo", "xbox", "console"], "whole_word": False},
    {"label": "Console", "words": ["ps4", "ps5"], "whole_word": True},
    {"label": "Network", "words": ["router", "access point", "wi-fi", "wifi", "mesh", "modem", "rj-11", "rj-45", "rj11", "rj45", "network", "เราเตอร์"], "whole_word": False},
    {"label": "HomeAppliance", "words": ["ตู้เย็น", "เครื่องซักผ้า", "เครื่องปรับอากาศ", "ไมโครเวฟ", "หม้อทอด", "เตาอบ", "เครื่องดูดฝุ่น", "หม้อหุงข้าว", "กาต้มน้ำ", "appliance"], "whole_word": False},
    {"label": "Battery", "words": ["battery", "แบตเตอรี่", "แบตเตอรี"], "whole_word": False},
    {"label": "Software", "words": ["software", "license", "microsoft 365", "office 365", "antivirus"], "whole_word": False},
    {"label": "UPS", "words": ["ups"], "whole_word": True},
]

# เรียงจากเจาะจงไปกว้าง
PLATFORMS = [
    {"label": "AirPods", "words": ["airpod"], "whole_word": False},
    {"label": "AppleWatch", "words": ["apple watch", "applewatch", "watch ultra"], "whole_word": False},
    {"label": "iPhone", "words": ["iphone", "ไอโฟน"], "whole_word": False},
    {"label": "iPad", "words": ["ipad", "ไอแพด"], "whole_word": False},
    {"label": "MacBook", "words": ["macbook", "imac", "mac mini", "mac studio"], "whole_word": False},
    # แยก Samsung ออกจาก Galaxy — printer samsung ไม่ใช่ Galaxy
    {"label": "Galaxy", "words": ["galaxy"], "whole_word": False},
    {"label": "Samsung", "words": ["samsung"], "whole_word": False},
    {"label": "Xiaomi", "words": ["xiaomi", "redmi", "poco"], "whole_word": False},
    {"label": "Huawei", "words": ["huawei"], "whole_word": False},
    # Realme เป็นคนละแบรนด์กับ OPPO (ร่วมกลุ่ม BBK เฉย ๆ)
    {"label": "Realme", "words": ["realme"], "whole_word": False},
    {"label": "OPPO", "words": ["oppo"], "whole_word": False},
    {"label": "Vivo", "words": ["vivo"], "whole_word": True},
    {"label": "Nintendo", "words": ["nintendo"], "whole_word": False},
    {"label": "PlayStation", "words": ["playstation", "ps5", "ps4"], "whole_word": True},
    {"label": "Asus", "words": ["asus", "rog ", "zenbook", "vivobook", "tuf "], "whole_word": False},
    {"label": "Acer", "words": ["acer", "predator", "nitro"], "whole_word": False},
    {"label": "Lenovo", "words": ["lenovo", "thinkpad", "ideapad", "legion"], "whole_word": False},
    {"label": "HP", "words": ["hp ", "pavilion", "omen", "envy", "compaq"], "whole_word": False},
    {"label": "Dell", "words": ["dell", "alienware", "inspiron", "latitude"], "whole_word": False},
    {"label": "MSI", "words": ["msi"], "whole_word": True},
    {"label": "Microsoft", "words": ["microsoft", "surface"], "whole_word": False},
    {"label": "Logitech", "words": ["logitech", "logi "], "whole_word": False},
    {"label": "Razer", "words": ["razer"], "whole_word": False},
    {"label": "JBL", "words": ["jbl"], "whole_word": True},
    {"label": "Anker", "words": ["anker", "soundcore"], "whole_word": False},
    # เติมใหม่ — 50% ของแถวเคยได้ Generic เพราะแบรนด์พวกนี้ไม่มีในกฎ
    {"label": "Sony", "words": ["sony", "bravia", "cyber-shot", "vaio"], "whole_word": False},
    {"label": "LG", "words": ["lg "], "whole_word": False},
    {"label": "Canon", "words": ["canon"], "whole_word": False},
    {"label": "Garmin", "words": ["garmin"], "whole_word": False},
    {"label": "Brother", "words": ["brother"], "whole_word": False},
    {"label": "Epson", "words": ["epson"], "whole_word": False},
    {"label": "WD-Kingston", "words": ["kingston", "sandisk", "adata", "toshiba", "seagate", "western digital", "wd "], "whole_word": False},
    {"label": "Electrolux", "words": ["electrolux"], "whole_word": False},
]

# แก้ความกำกวม — ได้ประเภทแล้วค่อยดูบริบทอีกที
# เช่น ได้ Case แล้วเจอคำว่า atx/tower → เปลี่ยนเป็น PC Case
DISAMBIGUATE = [
    {"from": "Case", "words": ["atx", "tower", "chassis", "computer case", "pc case", "เคสคอม", "เคส pc", "case & cooling"], "to": "PC Case", "whole_word": False},
    {"from": "Case", "words": ["bag", "กระเป๋า", "sleeve", "pouch", "ซองใส่"], "to": "Bag", "whole_word": False},
    # ชื่อรุ่นนาฬิกาพูดถึงวัสดุ case / สี band ของตัวเรือน ไม่ใช่ขายเคสหรือสาย
    {"from": "Case", "words": ["watch series", "watch ultra", "watch se", "watch nike+", "smartwatch"], "to": "Smartwatch", "whole_word": False},
    {"from": "Strap", "words": ["watch series", "watch ultra", "smartwatch", "galaxy watch"], "to": "Smartwatch", "whole_word": False},
    # "Core i5" ในชื่อโน้ตบุ๊กคือสเปก ไม่ใช่ขายซีพียู · เช่นเดียวกับ GeForce ในชื่อ PC
    {"from": "CPU", "words": ["notebook", "laptop", "macbook", "surface", "thinkpad", "ideapad", "vaio"], "to": "Notebook", "whole_word": False},
    {"from": "CPU", "words": ["desktop", "all-in-one", "computer set", "pc set", "imac"], "to": "Desktop", "whole_word": False},
    {"from": "GraphicCard", "words": ["notebook", "laptop", "macbook"], "to": "Notebook", "whole_word": False},
    {"from": "GraphicCard", "words": ["desktop", "all-in-one", "computer set", "pc set"], "to": "Desktop", "whole_word": False},
    # "Magic Keyboard" ในชื่อ MacBook คือคีย์บอร์ดติดเครื่อง ไม่ใช่ขายคีย์บอร์ด
    {"from": "Keyboard", "words": ["macbook", "notebook", "laptop"], "to": "Notebook", "whole_word": False},
    # adapter แปลงสัญญาณ ไม่ใช่ที่ชาร์จ
    {"from": "Charger", "words": ["displayport", "hdmi", "dvi", "vga", "ethernet", "converter"], "to": "Adapter", "whole_word": False},
]

# ---------------------------------------------------------------------------
# เขียน YAML ให้สคริปต์ (predict_all / explore_flags) ใช้กฎชุดเดียวกัน
import yaml
_rules = {
    "types": [{"label": r["label"], "words": r["words"],
               "whole_word": r.get("whole_word", False)} for r in TYPES],
    "platforms": [{"label": r["label"], "words": r["words"],
                   "whole_word": r.get("whole_word", False)} for r in PLATFORMS],
    "disambiguate": [{"from": r["from"], "to": r["to"], "words": r["words"],
                      "whole_word": r.get("whole_word", False)} for r in DISAMBIGUATE],
}
with open(SCRIPTS / "type_platform.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(_rules, f, allow_unicode=True, sort_keys=False, width=120)

import importlib, type_platform as TP
importlib.reload(TP)          # ให้ TP อ่านกฎใหม่ที่เพิ่งเขียน

print(f"เขียนกฎแล้ว → {SCRIPTS / 'type_platform.yaml'}")
print(f"ประเภท {len(TYPES)} · แพลตฟอร์ม {len(PLATFORMS)} · แก้ความกำกวม {len(DISAMBIGUATE)}")
print("ประเภททั้งหมด:", " · ".join(dict.fromkeys(r["label"] for r in TYPES)))


## 5 · ดูว่ากฎจับอะไรได้บ้าง


In [ ]:
chk = ["iPhone 15 Case Clear", "เคส iPhone 15", "CASE ATX 002",
       "Power Supply Plenty ATX 500W", "Tomtoc Sleeve for MacBook Pro 14",
       "ASUS TUF Gaming RTX 4060 8GB", "Tengu Gaming Mouse Pad"]
for s in chk:
    print(f"{str(TP.classify(s)):28} | {s}")


## 6 · สร้าง label จากกฎ

`Type_Platform` = ประเภทนำหน้า ตามด้วยแบรนด์ เช่น **`Case iPhone`**
`Item_Type` = ประเภทล้วน ไม่แยกแบรนด์ — หมวดน้อยกว่า เทรนง่ายกว่า


In [ ]:
# ใช้กฎจาก Cell 4 สร้าง label
df = TP.add_columns(df)          # -> Item_Type · Item_Platform · Type_Platform

# ⭐ เริ่มที่ Item_Type ก่อนเสมอ
#    Type_Platform = ชนิด+แบรนด์ -> 700+ หมวด หลายหมวดมีไม่ถึง 20 แถว
#    Item_Type     = ชนิดล้วน    -> ~40 หมวด ตัวอย่างต่อหมวดมากกว่าสิบเท่า
#    อยากได้แบรนด์ด้วย ค่อยเอา Item_Platform มาต่อทีหลัง ไม่ต้องให้โมเดลเรียนพร้อมกัน
LABEL_COL = "Item_Type"          # เปลี่ยนเป็น "Type_Platform" ได้ถ้าอยากลอง

print(f"label ที่จะเทรน = {LABEL_COL}  ({df[LABEL_COL].nunique()} หมวด)")
print(df[LABEL_COL].value_counts().head(20).to_string())
print(f"\nประเภทล้วน {df.Item_Type.nunique()} · แบรนด์ {df.Item_Platform.nunique()}"
      f" · รวม {df.Type_Platform.nunique()}")

# กองที่ยังแยกไม่ได้เหลือเท่าไหร่ - ตัวเลขนี้คือสุขภาพของกฎ
_dev = (df.Item_Type == "Device").mean()
print(f"\nยังตก Device {_dev:.1%}" + ("  ✅" if _dev < 0.20 else "  ⚠️ กฎยังขาดชนิดสินค้า"))


## 7 · เตรียม X, y — และ normalize ตรงนี้

**`NORM` ต้องสร้างจาก `data` ที่กรองแล้ว ไม่ใช่ `df`**

`data` ถูก `reset_index(drop=True)` ทำให้ตำแหน่งไม่ตรงกับ `df` อีกต่อไป
ถ้าสร้าง `NORM` จาก `df` แล้วมา `iloc` ด้วยตำแหน่งของ `data` จะได้ข้อความของแถวอื่น
**label กับข้อความจะจับคู่ผิดทั้งชุด โมเดลเรียนขยะ (เคยเจอ acc 0.0997)**


In [8]:
vc = df[LABEL_COL].value_counts()
data = df[df[LABEL_COL].isin(vc[vc >= 5].index)].reset_index(drop=True)
# pandas 3 คืน ArrowExtensionArray ที่ sklearn index ด้วย array ไม่ได้
y = data[LABEL_COL].to_numpy(dtype=object)

# ⚠️ normalize ตรงนี้ ไม่ใช่ Cell 3 - ต้องสร้างจาก data ที่กรองแล้ว
# ถ้าสร้างจาก df แล้วมา iloc ด้วยตำแหน่งของ data จะหยิบผิดแถวทั้งหมด
NORM = {c: normalize(data[c]) for c in TEXT_COLS if c in data.columns}

print(f"{len(data):,} แถว · {pd.Series(y).nunique()} หมวด · label={LABEL_COL}")
print(f"ตัดหมวดที่มีน้อยกว่า 5 แถวออก {vc[vc<5].size} หมวด")
assert len(NORM["ItemName"]) == len(y), "NORM กับ y ยาวไม่เท่ากัน"
# ⚡ CPU ช้า? → data=data.sample(5000,random_state=42).reset_index(drop=True); y=data[LABEL_COL].to_numpy(dtype=object); NORM={c:normalize(data[c]) for c in TEXT_COLS if c in data.columns}


215,873 แถว · 233 หมวด · label=Type_Platform
ตัดหมวดที่มีน้อยกว่า 5 แถวออก 66 หมวด


## 8 · โหลด backbone (auto GPU/CPU)

In [9]:
import torch
from sentence_transformers import SentenceTransformer
device = "cuda" if torch.cuda.is_available() else "cpu"
backbone = ("BAAI/bge-m3" if device == "cuda"
            else "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embedder = SentenceTransformer(backbone, device=device)
print(f"backbone={backbone} · device={device}")


def build_text(idx, drop_extra=None):
    """ประกอบข้อความส่งเข้า embedder - ทำทั้งคอลัมน์รวด ไม่ใช่ทีละแถว

    idx        ตำแหน่งแถวที่ต้องการ (ชี้เข้า NORM ซึ่ง normalize ไว้แล้วใน Cell 3)
    drop_extra boolean array - True = ใช้แค่ ItemName สำหรับแถวนั้น
               None = ใช้ครบทุกคอลัมน์

    ใส่ป้ายชื่อฟิลด์นำหน้า (หมวด: ซับ: brand:) เพราะ LLM เข้าใจข้อความมีบริบท
    ดีกว่าเอามาต่อกันเฉย ๆ
    """
    out = NORM["ItemName"].iloc[idx].reset_index(drop=True)
    if drop_extra is None:
        drop_extra = pd.Series(False, index=out.index)
    else:
        drop_extra = pd.Series(np.asarray(drop_extra), index=out.index)

    for col, tag in (("CategoryName", "หมวด: "), ("SubCategoryName", "ซับ: "),
                     ("Brand", "brand: ")):
        if col not in NORM:
            continue
        v = NORM[col].iloc[idx].reset_index(drop=True)
        add = (v.str.strip() != "") & ~drop_extra      # ว่างหรือถูกสั่งตัด = ไม่ต่อ
        out = out.where(~add, out + " | " + tag + v)
    return out.tolist()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

backbone=BAAI/bge-m3 · device=cuda


## 9 · โมเดล 1 — Random-drop (predict ItemName ล้วน)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# เทียบ classifier 200k แถว: LinearSVC เร็วกว่า LR ~3 เท่าและคะแนนดีกว่า
# ⚠️ อย่าเอาตัวเลข f1 ในคอมเมนต์ไปอ้าง - มันขึ้นกับ LABEL_COL ที่ใช้ตอนนั้น
#    ครั้งล่าสุดที่ LABEL_COL='Type_Platform' (299 หมวด) ได้ Acc 0.8930 / MacroF1 0.7534
# LinearSVC ไม่มี predict_proba -> ใช้ confidence() ด้านล่างแทน
USE_SVC = True


def make_clf():
    return (LinearSVC(C=1.0, class_weight="balanced") if USE_SVC
            else LogisticRegression(max_iter=2000, C=10,
                                    class_weight="balanced", n_jobs=-1))


def confidence(clf, X):
    """คืน (คลาสที่ทำนาย, ความมั่นใจ 0-1) · ใช้ได้ทั้ง LR และ SVC"""
    if hasattr(clf, "predict_proba"):
        p = clf.predict_proba(X)
    else:
        d = clf.decision_function(X)
        e = np.exp(d - d.max(axis=1, keepdims=True))   # softmax บน margin
        p = e / e.sum(axis=1, keepdims=True)           # ไม่ใช่ความน่าจะเป็นจริง
    return clf.classes_[p.argmax(1)], p.max(1)


# ⭐ Random-drop · สุ่มปิดคอลัมน์เสริม 50% ให้โมเดลชินกับการมีแค่ ItemName
# ตอนใช้จริงสินค้าใหม่ยังไม่มีใคร key หมวด -> มีแค่ชื่อ
rng = np.random.default_rng(42)
POS = np.arange(len(data))
drop1 = rng.random(len(data)) < 0.5
t1 = build_text(POS, drop_extra=drop1)

X1 = embedder.encode(t1, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
Xtr, Xte, ytr, yte, itr, ite = train_test_split(
    X1, y, POS, test_size=0.2, random_state=42, stratify=y)
clf1 = make_clf().fit(Xtr, ytr)

# วัดผลด้วย ItemName ล้วน = สภาพเดียวกับตอนใช้งานจริง
tt = build_text(ite, drop_extra=np.ones(len(ite), dtype=bool))
p1 = clf1.predict(embedder.encode(tt, normalize_embeddings=True))
acc1, f1a = accuracy_score(yte, p1), f1_score(yte, p1, average="macro")
print(f"[M1] {type(clf1).__name__} Acc={acc1:.4f} MacroF1={f1a:.4f}")


In [ ]:
# สุ่มชื่อสินค้ามาลองทำนาย
# ⛔ ห้ามใส่ df = pd.read_csv(...) ตรงนี้ - มันจะทับ df ที่มี Item_Type/Type_Platform
#    อยู่แล้วจากเซลล์ 6 และทำให้ต้องย้อนไปรันใหม่ตั้งแต่ต้น
N = 1000
lists = df["ItemName"].sample(N, random_state=42).tolist()
print(f"สุ่มมา {len(lists)} ชื่อ · ตัวอย่าง 5 อันแรก")
for x in lists[:5]:
    print("   ", x[:78])


In [ ]:
# ทำนายด้วย clf1 (โมเดล ItemName ล้วน) - ต้อง normalize แบบเดียวกับตอนเทรน
items = lists
txt = normalize(pd.Series(items)).tolist()
X = embedder.encode(txt, batch_size=64, normalize_embeddings=True,
                    show_progress_bar=len(txt) > 500)

pred, conf = confidence(clf1, X)          # ได้ความมั่นใจมาด้วย จะได้รู้ว่าอันไหนน่าสงสัย
out = pd.DataFrame({"ItemName": items, "result": pred, "conf": conf.round(4)})

OUT = SCRIPTS / "_out"
OUT.mkdir(exist_ok=True)
out.to_csv(OUT / f"sample{len(items)}.csv", index=False, encoding="utf-8-sig")

lo = out[out.conf < 0.5]
print(f"เขียน {OUT / f'sample{len(items)}.csv'}")
print(f"ความมั่นใจต่ำกว่า 0.5: {len(lo):,} แถว ({len(lo)/len(out):.1%}) <- ควรให้คนตรวจก่อน")
out.head(15)


In [ ]:
# (เซลล์นี้เคยเซฟโมเดลซ้ำกับเซลล์ 11 · ย้ายการเซฟไปไว้ที่นั่นที่เดียว)
# ดูว่าโมเดลทายอะไรเป็นส่วนใหญ่ - ถ้ากองใดกองหนึ่งบวมผิดปกติแปลว่ากฎยังมีปัญหา
out.result.value_counts().head(15)


## 10 · โมเดล 2 — Full (predict ครบ)

In [ ]:
t2 = build_text(POS)                    # ครบทุกคอลัมน์ ไม่ตัดอะไร
X2 = embedder.encode(t2, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
clf2 = make_clf().fit(X2[itr], ytr)
p2 = clf2.predict(X2[ite])
acc2, f2a = accuracy_score(yte, p2), f1_score(yte, p2, average="macro")
print(f"[M2] {type(clf2).__name__} Acc={acc2:.4f} MacroF1={f2a:.4f}")


Batches:   0%|          | 0/3374 [00:00<?, ?it/s]

## 11 · เปรียบเทียบ + บันทึก

In [ ]:
print(pd.DataFrame({"Model":["1.Random-drop (ItemName ล้วน)","2.Full (ครบทุกคอลัมน์)"],
                    "Accuracy":[acc1,acc2],"Macro-F1":[f1a,f2a]}).to_string(index=False))

# เก็บกฎไปกับโมเดลด้วย ไม่งั้นวันหน้าไม่รู้ว่าโมเดลนี้เทรนด้วยกฎชุดไหน
meta = {"backbone": backbone, "label_col": LABEL_COL,
        "types": TYPES, "platforms": PLATFORMS, "disambiguate": DISAMBIGUATE,
        "classes": sorted(set(y))}
# เซฟลง _out/ เพราะ predict.py มองหาที่นั่น (DEFAULT_MODEL = _out/model1_itemname.joblib)
OUT = SCRIPTS / "_out"
OUT.mkdir(exist_ok=True)
joblib.dump({**meta, "clf": clf1}, OUT / "model1_itemname.joblib")
joblib.dump({**meta, "clf": clf2}, OUT / "model2_full.joblib")
print(f"saved 2 models -> {OUT} · label={LABEL_COL} · {len(meta['classes'])} หมวด")


## 12 · ⭐ ทำ label สะอาด — จับ item ที่ key ผิด

ML ทายต่างจาก label เดิม + มั่นใจสูง = น่าจะ key ผิด → export ให้คนตรวจ → แก้ → retrain
(ขั้นนี้คุ้มกว่า fine-tune — ทำก่อน)

> **ใช้ `confidence()` ไม่ใช่ `predict_proba` ตรง ๆ** — `LinearSVC` ไม่มี `predict_proba` · ฟังก์ชันนี้ทำ softmax บน margin ให้แทน
> ค่าที่ได้ **ใช้เรียงลำดับได้ แต่ไม่ใช่ความน่าจะเป็นจริง** จึงตัดด้วย quantile แทนเลขตายตัว


In [ ]:
from sklearn.model_selection import StratifiedKFold

# ⛔ ของเดิมเขียน confidence(clf1, X1) ซึ่งเป็น in-sample
#    clf1 fit ด้วย 80% ของ X1 มาแล้ว เอามาทายตัวเองมันก็เห็นด้วยกับ label เกือบหมด
#    -> suspect จับได้แค่ 20% ที่เป็น test ส่วนที่เหลือเงียบทั้งที่อาจ key ผิด
# ✅ ทายแบบ out-of-fold: ทุกแถวถูกทายด้วยโมเดลที่ไม่เคยเห็นแถวนั้นมาก่อน
#    ราคา = fit 3 ครั้ง แต่ได้ทั้ง pred และ conf ที่เชื่อถือได้ทั้งชุด
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
pred = np.empty(len(y), dtype=object)
conf = np.zeros(len(y))
for k, (tr, te) in enumerate(skf.split(X1, y), 1):
    p, c = confidence(make_clf().fit(X1[tr], y[tr]), X1[te])
    pred[te], conf[te] = p, c
    print(f"  fold {k}/3 เสร็จ")

data["Cat_pred"], data["conf"] = pred, conf

# LinearSVC คืน margin ไม่ใช่ความน่าจะเป็น เลข 0.80 ตายตัวจึงใช้ไม่ได้
Q = 0.60
CUT = float(np.quantile(conf, Q))
suspect = (data[(data.Cat_pred != data[LABEL_COL]) & (data.conf >= CUT)]
           .sort_values("conf", ascending=False))
print()
print(f"label = {LABEL_COL} · เกณฑ์ความมั่นใจ >= {CUT:.3f}  (quantile {Q})")
print(f"น่าสงสัยว่า key ผิด: {len(suspect):,} จาก {len(data):,} ({len(suspect)/len(data):.2%})")

OUT = SCRIPTS / "_out"
OUT.mkdir(exist_ok=True)
cols = [c for c in ["ItemName", "CategoryName", LABEL_COL, "Cat_pred", "conf"]
        if c in suspect.columns]
suspect[cols].to_csv(OUT / "suspect.csv", index=False, encoding="utf-8-sig")
suspect[["ItemName", LABEL_COL, "Cat_pred", "conf"]].head(20)


## 13 · Fine-tune backbone — ทำท้ายสุด ถ้า baseline ยังไม่พอ

### ต่างจากของเดิมตรงไหน

```
เดิม   ItemName ─► [ backbone ] ─► เวกเตอร์ ─► [ LinearSVC ] ─► หมวด
                    ❄️ แช่แข็ง                  🔥 เรียนตรงนี้

ใหม่   ItemName ─► [ backbone ] ─► เวกเตอร์ ─► [ LinearSVC ] ─► หมวด
                    🔥 ปรับด้วย                 🔥 เรียนใหม่
```

**เปลี่ยนแค่ชั้นแรก** — classifier ยังเป็น `LinearSVC` ตัวเดิม พารามิเตอร์เดิม test ชุดเดิม seed เดิม
ถ้าเปลี่ยนสองชั้นพร้อมกันจะแยกไม่ออกว่าที่ดีขึ้น/แย่ลงมาจากไหน

---

### 🛑 เช็ก 3 ข้อนี้ก่อน ไม่ครบอย่าเพิ่งรัน

| # | เงื่อนไข | เช็กยังไง |
|---|---|---|
| 1 | **มี GPU** | Cell 1 ต้องพิมพ์ชื่อการ์ด ไม่ใช่ `ไม่มี - จะใช้ CPU` · CPU ช้าจนไม่คุ้ม |
| 2 | **`Device` ต่ำกว่า 20%** | บรรทัดท้าย Cell 6 · ถ้ายังสูงแปลว่ากฎขาดชนิดสินค้า **ไปแก้ Cell 4 ก่อน** |
| 3 | **`LABEL_COL = "Item_Type"`** | Cell 6 · `Type_Platform` มี 700+ หมวด triplet loss หา positive แทบไม่ได้ |

> ⚠️ **label มาจากกฎที่เราเขียนเอง** — fine-tune คือสอนให้ embedding แยกตาม label นั้น
> **label ผิด = สอนให้แยกผิดแม่นขึ้น** นี่คือเหตุผลที่ข้อ 2 กับ 3 สำคัญกว่าตัว fine-tune เอง

---

### ขั้นตอน

**1 · ตั้ง `RUN_FT = True`** ในเซลล์ถัดไป แล้วรัน (ค่าอื่นปล่อยไว้ก่อน)

**2 · รอ** — `mini` ที่ 20,000 แถวบน T4 ประมาณ 10-15 นาที · `bge` ~40 นาที และอาจ OOM ถ้า batch 32

**3 · อ่านตารางท้ายเซลล์**

```
                    accuracy    f1_macro
frozen                0.xxxx      0.xxxx
fine-tuned            0.xxxx      0.xxxx
ต่าง                 +0.xxxx     +0.xxxx
```

**ดู `f1_macro` ไม่ใช่ `accuracy`** — หมวดเบ้มาก accuracy บอกอะไรไม่ได้

| `f1_macro` ต่าง | แปลว่า | ทำต่อ |
|---|---|---|
| **> +0.01** | ✅ คุ้ม | ข้อ 4 |
| **−0.01 ถึง +0.01** | ⚠️ ไม่คุ้ม | ใช้แบบแช่แข็งต่อไป · ไปทำ label สะอาด (Cell 12) |
| **< −0.01** | 🛑 แย่ลง | **label ยัง noisy** กลับไปแก้กฎ Cell 4 |

**4 · เอา backbone ใหม่ไปใช้** — ถ้าคุ้ม แก้ Cell 8 แล้วรัน Cell 8-11 ใหม่

```python
backbone = str(SCRIPTS / "_out" / "Item_Type-finetuned")
```

**5 · ค่อยขยับทีละอย่าง** ถ้าอยากดันต่อ

```
N 20000 → 50000   →   model mini → bge (--batch 16)   →   epochs 1 → 2
```

**อย่าเพิ่ม epoch หรือเปลี่ยนโมเดลใหญ่ขึ้นเป็นอย่างแรก** — มักไม่ใช่สาเหตุ
คอขวดที่เจอบ่อยกว่าคือ label และจำนวนตัวอย่างต่อหมวด

---

### ทำไมต้อง `BatchAllTripletLoss` + `GROUP_BY_LABEL`

| loss | ต้องเตรียมข้อมูลยังไง | เหมาะไหม |
|---|---|---|
| `CosineSimilarityLoss` | คู่ `(a, b, คะแนน 0-1)` | ❌ ต้องกำหนดคะแนนความเหมือนเอง ซึ่งไม่มีข้อมูล |
| `MultipleNegativesRankingLoss` | คู่ `(anchor, positive)` | ⚠️ ต้องสร้างคู่เอง |
| **`BatchAllTripletLoss`** | **`(ประโยค, label)` เฉย ๆ** | ✅ **มี label อยู่แล้ว** |

มันเรียนแบบนี้

```
ในแต่ละ batch หยิบ 3 ตัว
    anchor     "iphone 15 case clear"     หมวด Case
    positive   "เคส iphone 14 ใส"          หมวดเดียวกัน  ← ดันให้ใกล้
    negative   "iphone 15 pro max 256gb"  คนละหมวด      ← ดันให้ไกล
```

**"All" = ใช้ทุกคู่ที่เป็นไปได้ใน batch** ไม่ใช่สุ่มมาบางคู่

🔑 **`batch_sampler=BatchSamplers.GROUP_BY_LABEL` ขาดไม่ได้**
ถ้า batch มีหมวดละ 1 ตัว จะสร้าง positive ไม่ได้เลย → `loss = 0` → **รันผ่านแต่ไม่เรียนอะไร**

รายละเอียดเต็ม → Obsidian: `ITEC Model - Fine-tune Design`


In [ ]:
# ============================================================================
# 13 · Fine-tune backbone — เปลี่ยนแค่ชั้นแรก classifier ยังเป็นตัวเดิม
# ============================================================================
#   เดิม  ItemName ─► [backbone ❄️แช่แข็ง] ─► เวกเตอร์ ─► [LinearSVC 🔥] ─► หมวด
#   ใหม่  ItemName ─► [backbone 🔥ปรับด้วย] ─► เวกเตอร์ ─► [LinearSVC 🔥] ─► หมวด
#
# เปลี่ยนทีละชั้นเพื่อให้พิสูจน์ได้ว่าอะไรทำให้ดีขึ้น
# ถ้าเปลี่ยนสองชั้นพร้อมกันจะแยกไม่ออกว่าผลมาจากไหน
#
# ⚠️ label มาจากกฎที่เราเขียนเอง — fine-tune = สอนให้แยกตาม label นั้น
#    label ผิด = สอนให้ผิดแม่นขึ้น ถ้าผลออกมา "ไม่ช่วย" ก็เป็นคำตอบที่มีค่า
#    เพราะพิสูจน์ว่าคอขวดไม่ได้อยู่ที่ embedding แต่อยู่ที่ label

# ----------------------------------------------------------------------------
# ขั้นตอน (ย่อ · รายละเอียดอยู่ในเซลล์ markdown ด้านบน)
#   0) เช็กก่อน : มี GPU · Device% < 20 · LABEL_COL = "Item_Type"
#   1) ตั้ง RUN_FT = True แล้วรันเซลล์นี้ · ค่าอื่นปล่อยไว้ก่อน
#   2) รอ ~10-15 นาที (mini/20k บน T4) · bge ~40 นาที และอาจ OOM ถ้า batch 32
#   3) อ่านตารางท้ายเซลล์ ดู "f1_macro ต่าง" ไม่ใช่ accuracy
#         > +0.01  ✅ คุ้ม        -> ไปข้อ 4
#         ±0.01    ⚠️ ไม่คุ้ม     -> ใช้แบบแช่แข็งต่อ ไปทำ label สะอาด (Cell 12)
#         < -0.01  🛑 แย่ลง      -> label ยัง noisy กลับไปแก้กฎ Cell 4
#   4) ถ้าคุ้ม แก้ Cell 8 เป็น
#         backbone = str(SCRIPTS / "_out" / "Item_Type-finetuned")
#      แล้วรัน Cell 8-11 ใหม่
#   5) อยากดันต่อ ค่อยขยับทีละอย่าง: N 20k->50k  ->  mini->bge  ->  epochs 1->2
#      อย่าเพิ่ม epoch เป็นอย่างแรก คอขวดมักอยู่ที่ label ไม่ใช่โมเดล
# ----------------------------------------------------------------------------

RUN_FT   = False          # ⬅️ เปลี่ยนเป็น True เมื่อพร้อมรันจริง (กัน Run-All เผลอรัน)
FT_N     = 20000          # 20k พอบอกทิศทางแล้ว อย่ารันเต็ม 216k ตั้งแต่รอบแรก
FT_EPOCH = 1              # หลายหมวดตัวอย่างน้อย เกิน 1 เสี่ยงจำข้อมูลแทนเรียน pattern
FT_BATCH = 32             # ต้องใหญ่พอให้มีหลายหมวดต่อ batch ไม่งั้นสร้าง triplet ไม่ได้
FT_LR    = 2e-5           # มาตรฐานของ transformer · สูงกว่านี้ทำลายของที่โมเดลรู้มาเดิม
FT_SEQ   = 64             # ชื่อสินค้าสั้น ~20 token · default 512 เปลืองเปล่า
FT_MINPC = 4              # หมวดที่มีน้อยกว่านี้ตัดทิ้ง สร้าง positive ไม่ได้

if not RUN_FT:
    print("ข้าม fine-tune · ตั้ง RUN_FT = True แล้วรันเซลล์นี้ใหม่เมื่อพร้อม")
else:
    # Colab/Kaggle มักไม่มี datasets/accelerate ติดมา ลงให้ก่อน
    for _pkg, _mod in (("datasets", "datasets"), ("accelerate", "accelerate")):
        try:
            __import__(_mod)
        except ImportError:
            print(f"ติดตั้ง {_pkg} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)

    import time
    import torch
    from datasets import Dataset
    from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                       SentenceTransformerTrainingArguments)
    from sentence_transformers.losses import BatchAllTripletLoss
    from sentence_transformers.training_args import BatchSamplers
    from sklearn.model_selection import train_test_split

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    assert dev == "cuda", "fine-tune บน CPU ช้าจนไม่คุ้ม — ไปรันบน Kaggle T4"

    # ── เตรียมข้อมูล · ใช้ ItemName ล้วน = สภาพเดียวกับตอนใช้งานจริง ──────
    _sub = data.sample(min(FT_N, len(data)), random_state=42).reset_index(drop=True)
    _vc = _sub[LABEL_COL].value_counts()
    _sub = _sub[_sub[LABEL_COL].isin(_vc[_vc >= FT_MINPC].index)].reset_index(drop=True)

    ft_txt = normalize(_sub["ItemName"]).tolist()
    ft_y = _sub[LABEL_COL].to_numpy(dtype=object)
    tr, te = train_test_split(np.arange(len(_sub)), test_size=0.2,
                              random_state=42, stratify=ft_y)
    print(f"label={LABEL_COL} · {len(_sub):,} แถว · {pd.Series(ft_y).nunique()} หมวด"
          f" · train {len(tr):,} / test {len(te):,}")
    print(f"ตัวอย่างต่อหมวดเฉลี่ย {len(_sub)/pd.Series(ft_y).nunique():.0f}"
          "   <- ยิ่งน้อย triplet loss ยิ่งหา positive ไม่เจอ")

    def ft_score(m, tag):
        """encode -> LinearSVC ตัวเดิม -> วัดบน test ชุดเดิม"""
        V = m.encode(ft_txt, batch_size=64, normalize_embeddings=True,
                     show_progress_bar=True)
        c = make_clf().fit(V[tr], ft_y[tr])
        p = c.predict(V[te])
        a, f = accuracy_score(ft_y[te], p), f1_score(ft_y[te], p, average="macro")
        print(f"  [{tag}] acc={a:.4f} f1_macro={f:.4f}")
        return a, f

    # ── ① baseline · backbone แช่แข็ง ───────────────────────────────────
    print("\n① baseline · backbone แช่แข็ง")
    _base = SentenceTransformer(backbone, device=dev)
    _base.max_seq_length = FT_SEQ          # ต้องเท่ากับตัว fine-tune ไม่งั้นเทียบไม่ยุติธรรม
    acc0, f10 = ft_score(_base, "frozen")

    # ── ② fine-tune · เห็นแต่ train เท่านั้น ────────────────────────────
    # ถ้าให้เห็น test ด้วย คะแนนจะสวยแบบหลอกตัวเอง เพราะโมเดลจำคำตอบไว้แล้ว
    print(f"\n② fine-tune · {FT_EPOCH} epoch · batch {FT_BATCH} · lr {FT_LR}")
    _codes, _ = pd.factorize(pd.Series(ft_y[tr]))        # triplet loss ต้องการ label เป็น int
    ds = Dataset.from_dict({"sentence": [ft_txt[i] for i in tr],
                            "label": _codes.tolist()})

    ft_model = SentenceTransformer(backbone, device=dev)
    ft_model.max_seq_length = FT_SEQ
    loss = BatchAllTripletLoss(ft_model)

    OUT = SCRIPTS / "_out"
    OUT.mkdir(exist_ok=True)
    targs = SentenceTransformerTrainingArguments(
        output_dir=str(OUT / "ft-ckpt"),
        num_train_epochs=FT_EPOCH,
        per_device_train_batch_size=FT_BATCH,
        learning_rate=FT_LR,
        warmup_ratio=0.1,                # ค่อย ๆ เพิ่ม lr 10% แรก กันโมเดลพังตั้งแต่ก้าวแรก
        fp16=True,
        # ⭐ ขาดบรรทัดนี้ fine-tune จะรันผ่านแต่ไม่เรียนอะไรเลย
        #    ถ้า batch มีหมวดละ 1 ตัว จะสร้าง positive ไม่ได้ -> loss = 0
        batch_sampler=BatchSamplers.GROUP_BY_LABEL,
        logging_steps=100,
        save_strategy="no",
        report_to=[],
    )
    _t0 = time.time()
    SentenceTransformerTrainer(model=ft_model, args=targs,
                               train_dataset=ds, loss=loss).train()
    print(f"  ใช้เวลา {time.time()-_t0:.0f}s")

    ft_dir = OUT / f"{LABEL_COL}-finetuned"
    ft_model.save(str(ft_dir))
    print(f"  เซฟ backbone -> {ft_dir}")

    # ── ③ วัดใหม่ · test ชุดเดิม seed เดิม classifier เดิม ───────────────
    print("\n③ หลัง fine-tune")
    acc1, f11 = ft_score(ft_model, "fine-tuned")

    # ── สรุป · ดู f1_macro ไม่ใช่ accuracy เพราะหมวดเบ้มาก ───────────────
    d_acc, d_f1 = acc1 - acc0, f11 - f10
    print("\n" + "=" * 58)
    print(f"{'':20}{'accuracy':>12}{'f1_macro':>12}")
    print(f"{'frozen':20}{acc0:>12.4f}{f10:>12.4f}")
    print(f"{'fine-tuned':20}{acc1:>12.4f}{f11:>12.4f}")
    print(f"{'ต่าง':20}{d_acc:>+12.4f}{d_f1:>+12.4f}")
    print("=" * 58)
    print("✅ คุ้ม · ใช้ backbone ที่ปรับแล้ว" if d_f1 > 0.01 else
          "⚠️ ไม่คุ้ม · label ยังไม่สะอาดพอ ใช้แบบแช่แข็งต่อไป (ดู Cell 12)"
          if d_f1 > -0.01 else
          "🛑 แย่ลง · label noisy เกินไป ต้องกลับไปแก้กฎก่อน")

    pd.DataFrame([{"backbone": backbone, "label": LABEL_COL, "n": len(_sub),
                   "epochs": FT_EPOCH, "acc_frozen": acc0, "f1_frozen": f10,
                   "acc_ft": acc1, "f1_ft": f11, "d_f1": d_f1}]
                 ).to_csv(OUT / "finetune_result.csv", index=False, encoding="utf-8-sig")
    print(f"เขียน {OUT / 'finetune_result.csv'}")
    print(f"\nเอาไปใช้ต่อ: แก้ Cell 8 เป็น  backbone = r\"{ft_dir}\"  แล้วรัน Cell 8-11 ใหม่")


### 📋 บันทึกผลทุกรอบ — ไม่งั้นสัปดาห์หน้าจำไม่ได้ว่าลองอะไรไปแล้ว

เซลล์บนเขียนต่อท้าย `_out/finetune_result.csv` ให้อยู่แล้ว เปิดดูย้อนหลังได้

| backbone | label | n | epochs | f1_frozen | f1_ft | d_f1 | สรุป |
|---|---|---|---|---|---|---|---|
| mini | Item_Type | 20000 | 1 | | | | |
| bge | Item_Type | 20000 | 1 | | | | |

### ถ้า fine-tune ไม่ช่วย — เรียงตามความคุ้ม

| ลองตามลำดับ | เหตุผล |
|---|---|
| 1. **แก้กฎ Cell 4 ให้ `Device` ต่ำลงอีก** | label ผิดเป็นคอขวดที่ใหญ่กว่า embedding มาก |
| 2. **ตรวจ `_out/suspect.csv` ด้วยตาคน** | Cell 12 ชี้แถวที่ ML ไม่เห็นด้วยกับกฎแบบมั่นใจ = จุดที่กฎน่าจะผิด |
| 3. `N = 50000` | ตัวอย่างต่อหมวดมากขึ้น triplet มีคู่ให้เลือกมากขึ้น |
| 4. `FT_MINPC = 10` | ตัดหมวดที่ตัวอย่างน้อยเกินไปออก |
| 5. `--model bge --batch 16` | ค่อยขยับมาตัวใหญ่เป็นอย่างสุดท้าย |

> 🔑 **ตัวเลขทุกตัวในโน้ตบุ๊กนี้วัด "เหมือนกฎแค่ไหน" ไม่ใช่ "ถูกแค่ไหน"**
> อยากรู้ว่าถูกจริงต้องมีคนตรวจ gold set ~300 แถว (สุ่มแบบคุมสัดส่วน 150 + จาก `suspect.csv` 150)
> ก่อนมี gold set ตัวเลข `f1_macro` ใช้เทียบรอบต่อรอบได้ แต่ห้ามเอาไปอ้างว่า "ระบบแม่น 95%"

### รันนอกโน้ตบุ๊กก็ได้ — โปรเจกต์แยก `scripts/itec/finetune/`

```bash
python sync_rules.py                                    # ดึงกฎจาก ../category/ มาก่อนเสมอ
python finetune.py --label Item_Type -n 20000 --model mini
python finetune.py --label Item_Type -n 20000 --model bge --batch 16
```

ตัวนั้นทำสิ่งเดียวกันทุกอย่าง ต่างแค่โหลด CSV เองและ `add_columns` เอง
**กฎมีที่เดียวคือ `../category/` เสมอ** — `type_platform.py` / `.yaml` ในโฟลเดอร์ fine-tune เป็นสำเนา ห้ามแก้ที่นั่น


---

# ส่วนที่ 2 · งานที่เหลือ รวมไว้ที่นี่

ส่วนบน (Cell 1-13) คือ **การออกแบบหมวดใหม่ของเราเอง** — `MY_TAXONOMY` 9 หมวด

ส่วนนี้คือ **การทำงานกับกฎของ SQL production** ที่ให้ผล 5 คอลัมน์

| | กฎ | ผลลัพธ์ |
|---|---|---|
| ส่วนบน | `FLAG_RULES` + `MY_TAXONOMY` ในโน้ตบุ๊ก | `MyCategory` 9 หมวด |
| **ส่วนนี้** | `item_keywords.yaml` + `rules.py` (67 flag จาก SQL) | `Sale_Type` · `Product_Dimension` · `Product_Purpose` · `Main_Product_Dimension` · `Sub_Product_Dimension` |

**กฎอยู่ในไฟล์ `.py` ที่เดียว โน้ตบุ๊กแค่เรียกใช้** — แก้กฎให้แก้ที่ไฟล์ ไม่ใช่ที่นี่


## A · เตรียมสคริปต์

In [ ]:
def run(script, *args):
    """เรียกสคริปต์แล้วพิมพ์ผล · SCRIPTS ถูกตั้งไว้ตั้งแต่ Cell 1 แล้ว"""
    r = subprocess.run([sys.executable, str(SCRIPTS / script), *map(str, args)],
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(r.stdout[-6000:])
    if r.returncode:
        print("--- stderr ---"); print(r.stderr[-2000:])
    return r.returncode

print("SCRIPTS =", SCRIPTS.resolve())
print("ไฟล์ที่เจอ:", sorted(p.name for p in SCRIPTS.glob("*.py")))

## B · สำรวจ flag — ควรเพิ่ม/ลดตัวไหน

ออก 4 ไฟล์ · ผลที่วัดไว้แล้ว **กอง `Accessory and Others` = 31.6% ของทั้งหมด**
→ [[ITEC Flag & Category Review Guide]] ใน Obsidian

In [ ]:
run("explore_flags.py", "--csv", "dim_item_itec.csv")

## C · ออกผล 5 คอลัมน์ตาม SQL

`ItemName → TF-IDF → 50 binary classifier → flags → rules.py → 5 คอลัมน์`

**ML แทนที่แค่ขั้น keyword matching** กฎที่เหลือไม่แตะ ผลจึงไม่ขัดกันเอง

In [ ]:
run("predict_all.py", "--csv", "dim_item_itec.csv",
    "-n", "216009", "--out", "itec_categorized.csv", "--save")

### ดูแถวที่ ML เห็นไม่ตรงกับกฎเดิม

**คอลัมน์ `differs` คือของที่มีค่าที่สุด** — ไม่ใช่ error แต่คือจุดที่ต้องให้คนตัดสินว่าใครถูก

In [ ]:
res = pd.read_csv("itec_categorized.csv", encoding="utf-8-sig")
d = res[res.differs]
print(f"ต่างจากกฎเดิม {len(d):,} / {len(res):,} ({len(d)/len(res):.2%})\n")
print(d.groupby(["rule_Main_Product_Dimension","Main_Product_Dimension"])
        .size().sort_values(ascending=False).head(15).to_string())
d[["ItemName","rule_Main_Product_Dimension","Main_Product_Dimension"]].head(20)

## D · เทียบ backbone และ classifier

ผลที่วัดบน CPU ไว้แล้ว 200,000 แถว

| | accuracy | f1_macro | เวลา |
|---|---|---|---|
| TF-IDF + **LinearSVC** | 0.9629 | **0.9519** | 115s |
| TF-IDF + LogisticRegression | 0.9522 | 0.9336 | 384s |
| TF-IDF + ComplementNB | 0.8705 | 0.8193 | 57s |
| TF-IDF + SGD log_loss | 0.8993 | 0.7757 | 89s |

**ที่ยังไม่รู้คือ `bge-m3` ชนะ 0.9519 ได้ไหม** — cell ถัดไปตอบข้อนี้ (ต้องมี GPU ถึงจะเร็ว)

In [ ]:
run("benchmark_models.py", "--csv", "dim_item_itec.csv", "-n", "5000")

### เทียบเฉพาะ classifier (ไม่ใช้ embedding · เร็ว)

In [ ]:
run("benchmark_models.py", "--csv", "dim_item_itec.csv",
    "-n", "50000", "--skip-embed", "--compare-clf")

## E · ดาวน์โหลดผลทั้งหมด

⚠️ **Colab หลุดแล้วไฟล์หาย** — ต้องโหลดก่อนปิด

In [ ]:
WANT = ["itec_categorized.csv", "suspect.csv", "flag_usage.csv",
        "accessory_words.csv", "unmatched_words.csv", "flag_overlap.csv",
        "flag_scores.csv", "benchmark_result.csv",
        "itec_flag_models.joblib", "model1_itemname.joblib", "item_embeddings.npy"]

have = [f for f in WANT if Path(f).exists()]
print("มี:", have)
print("ไม่มี:", [f for f in WANT if f not in have])

if IN_COLAB:
    import zipfile
    with zipfile.ZipFile("itec_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for f in have: z.write(f)
    from google.colab import files
    files.download("itec_results.zip")      # รวมเป็นไฟล์เดียว โหลดทีเดียวจบ

---

## สรุปว่าแต่ละ cell ทำอะไร

| ส่วน | ทำอะไร | ต้อง GPU |
|---|---|---|
| Cell 1-13 | ออกแบบหมวดใหม่ + เทรนโมเดล `MyCategory` | ช่วยให้เร็ว |
| **B** | สำรวจ flag → 4 ไฟล์ไว้แยกมือ | ❌ |
| **C** | ออก 5 คอลัมน์ตาม SQL + หาแถวที่ต่างจากกฎ | ❌ |
| **D** | เทียบ backbone / classifier | ช่วยให้เร็ว |
| **E** | รวมผลเป็น zip แล้วโหลด | ❌ |

**B · C · E ไม่ต้องใช้ GPU เลย** รันบนเครื่องตัวเองได้ ถ้าไม่อยากรอโควตา
